In [ ]:
# Quantum Circuit Visualization (QCV)
# Author: Armaan Kautish
# Last Updated: July 4, 2025

# Import necessary libraries.
import sys
import matplotlib.pyplot as plt
from itertools import product

# Try to import the sQUlearn library. If it's not installed,
# print a helpful error message and exit the script.
try:
    from squlearn.encoding_circuit import (
        HubregtsenEncodingCircuit,
        ChebyshevPQC,
        YZ_CX_EncodingCircuit,
        HighDimEncodingCircuit,
        KyriienkoEncodingCircuit,
        ParamZFeatureMap,
        ChebyshevRx
    )
except ImportError:
    print("Error: 'squlearn' library not found.")
    print("Please install it using: pip install squlearn")
    sys.exit(1)


# This function asks the user for a list of inputs (like "1, 2, 3").
# It's a general-purpose function we can reuse for different questions.
def get_multiple_inputs(prompt, type_converter, validation_func, error_msg):
    # Keep asking until we get valid input.
    while True:
        try:
            # Get the user's input as a string.
            raw_input = input(prompt)
            # Split the string by commas and remove any extra spaces.
            items_str = [item.strip() for item in raw_input.split(',')]
            # Convert each item from a string to the type we want (e.g., an integer).
            values = [type_converter(item) for item in items_str]
            
            # Check if all the values are valid according to the rules we set.
            if all(validation_func(v) for v in values):
                return values  # If everything is good, return the list of values.
            else:
                print(error_msg) # Otherwise, print the specific error message.
        except (ValueError, TypeError):
            # If the user enters something that can't be converted (like text instead of a number),
            # print the error message and the loop will ask again.
            print(error_msg)

# This function just asks a simple yes or no question.
def get_yes_no_input(prompt):
    while True:
        # Get the input and make it lowercase to handle 'Y' or 'N'.
        choice = input(prompt).lower().strip()
        if choice in ['y', 'n']:
            # Return True if the user entered 'y', and False if they entered 'n'.
            return choice == 'y'
        # If they entered something else, ask again.
        print("Invalid input. Please enter 'y' or 'n'.")

# This function generates and shows a single quantum circuit plot.
def generate_and_display_plot(circuit_class, name, qubits, layers, save_plot, file_format):
    print(f"\nGenerating: {name} with {qubits} Qubits, {layers} Layer(s)...")
    fig = None  # Start with fig as None in case an error happens early.
    try:
        # For these circuits, the number of features is the same as the number of qubits.
        num_features = qubits

        # Create an instance of the quantum circuit we want to visualize.
        circuit = circuit_class(
            num_qubits=qubits,
            num_features=num_features,
            num_layers=layers
        )

        # Create a plot figure. The size of the plot will change based on
        # how many qubits and layers there are, so it doesn't look too crowded.
        fig, ax = plt.subplots(figsize=(max(10, qubits * 2), max(6, layers * 2)))
        
        # Set the font size for the circuit drawing.
        style = {'fontsize': 12, 'subfontsize': 8}
        
        # This is the main command that draws the circuit.
        circuit.draw(output='mpl', ax=ax, style=style)
        
        # Create a nice title for the plot.
        title_str = (
            f"{name.replace('_', ' ').title()} Encoding Circuit\n"
            f"({qubits} Qubits, {layers} Layer{'s' if layers > 1 else ''})"
        )
        ax.set_title(title_str, fontsize=16, pad=20)
        
        # Adjust the layout to make sure nothing is cut off.
        plt.tight_layout()
        
        # Show the plot on the screen. The script will pause here until you close the plot window.
        print("Displaying plot. Close the plot window to continue...")
        plt.show()

        # If the user chose to save the plots...
        if save_plot:
            # Create a filename based on the circuit's properties.
            filename = f"{name}_Q{qubits}_L{layers}.{file_format}"
            # Save the figure to a file.
            fig.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"Success! Plot saved as '{filename}'")
        else:
            print("Plot displayed and closed. Not saving.")

    # If something goes wrong while trying to create the circuit...
    except Exception as e:
        # Print an error message. This usually happens if a circuit can't be
        # built with the number of qubits or layers requested (e.g., a 2-qubit
        # gate in a 1-qubit circuit).
        print(f"Error generating circuit '{name}' with {qubits} qubits and {layers} layers.")
        print(f"Reason: {e}")
        print(f"(This combination may not be supported by the circuit's architecture.)")

    # The 'finally' block will always run, whether there was an error or not.
    finally:
        # This makes sure the plot figure is closed properly to free up memory.
        if fig:
            plt.close(fig)


# This is the main function that runs the whole visualization tool.
def visualize_circuits_batch():
    # This dictionary holds information about each available circuit,
    # like its class name and the minimum number of qubits it needs.
    ENCODING_INFO = {
        "hubregtsen":   {"class": HubregtsenEncodingCircuit, "min_qubits": 1},
        "chebyshev":    {"class": ChebyshevPQC,              "min_qubits": 2},
        "yz_cx":        {"class": YZ_CX_EncodingCircuit,     "min_qubits": 2},
        "highdim":      {"class": HighDimEncodingCircuit,    "min_qubits": 2},
        "kyriienko":    {"class": KyriienkoEncodingCircuit,  "min_qubits": 2},
        "paramz":       {"class": ParamZFeatureMap,          "min_qubits": 1},
        "chebyshev_rx": {"class": ChebyshevRx,               "min_qubits": 1}
    }
    
    # Ask the user which circuits they want to see.
    print("\n--- Quantum Circuit Batch Visualization Tool ---")
    print("Available Encoding Circuits (min layers is 1 for all):")
    circuit_options = list(ENCODING_INFO.keys())
    # Show a numbered list of the available circuits.
    for i, name in enumerate(circuit_options):
        min_q = ENCODING_INFO[name]["min_qubits"]
        print(f"  {i+1}: {name} (min qubits: {min_q})")
    
    choice_prompt = "\nEnter circuit numbers, separated by commas (e.g., 1, 3, 4): "
    circuit_indices = get_multiple_inputs(
        choice_prompt, int, lambda x: 1 <= x <= len(circuit_options),
        f"Error: Please enter numbers between 1 and {len(circuit_options)}."
    )
    # Convert the user's number choices into a list of circuit names.
    chosen_circuits = [circuit_options[i - 1] for i in circuit_indices]

    # Ask the user for the number of qubits they want to try.
    qubit_prompt = "Enter number of qubits, separated by commas (e.g., 2, 4): "
    qubit_counts = get_multiple_inputs(
        qubit_prompt, int, lambda x: x > 0, "Error: Qubit counts must be positive integers."
    )
    
    # Ask the user for the number of layers they want to try.
    layer_prompt = "Enter number of layers, separated by commas (e.g., 1, 2, 3): "
    layer_counts = get_multiple_inputs(
        layer_prompt, int, lambda x: x > 0, "Error: Layer counts must be positive integers."
    )

    # Ask the user if they want to save the plots.
    save_plots = get_yes_no_input("\nDo you want to save the generated plots? (y/n): ")
    file_format = None
    if save_plots:
        # If they say yes, ask them which file format they want.
        format_choice = get_multiple_inputs(
            "Choose save format [1: PNG, 2: SVG, 3: PDF]: ",
            int, lambda x: x in [1, 2, 3], "Invalid choice. Enter 1, 2, or 3."
        )[0]
        formats = {1: 'png', 2: 'svg', 3: 'pdf'}
        file_format = formats[format_choice]

    # Use `itertools.product` to create a list of all possible combinations
    # of the user's choices (e.g., [('hubregtsen', 2, 1), ('hubregtsen', 2, 2), ...]).
    parameter_combinations = list(product(chosen_circuits, qubit_counts, layer_counts))
    
    print(f"\nFound {len(parameter_combinations)} unique circuit configurations to generate.")

    # Loop through every combination we created.
    for circuit_name, num_qubits, num_layers in parameter_combinations:
        # Get the circuit's class from our info dictionary.
        circuit_class = ENCODING_INFO[circuit_name]["class"]
        # Call the main plotting function for this combination.
        generate_and_display_plot(circuit_class, circuit_name, num_qubits, num_layers, save_plots, file_format)


# This is a standard Python practice. The code inside this `if` statement
# will only run when you execute the script directly (e.g., `python QCV.py`).
if __name__ == "__main__":
    visualize_circuits_batch()
    print("\n--- Batch visualization process finished. ---")